[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aecelaya/MIST/blob/main/examples/mist_data_loading_benchmark.ipynb)

# Where MIST's training time goes

This notebook times MIST's training loop on synthetic data shaped like two real
datasets:

- **LiTS-like:** one CT channel, about 512 × 512 × 480 voxels per case after
  resampling, labels 0–2 (a large organ with small lesions).
- **BraTS-like:** four MRI channels, about 140 × 172 × 138 voxels, labels 0–3.

It recreates a real LiTS run's settings: `nnunet-plain`, a 128³ patch, batch 2,
FP32, `torch.compile`, all augmentations, Dice + cross-entropy, AdamW, and the DALI
loader with 8 CPU workers. It then splits each training step into **waiting for
data** and **compute**, and tries the changes that might close the gap:

- **Preprocessed data.** *Old* is what MIST saved before its fix: labels in
  Fortran (column-major) order, made with MIST's old conversion code, and no saved
  foreground locations, so the generic loader scans each label. *New* is what
  `mist_preprocess` saves now: the same labels in C order, plus each case's
  foreground locations, made with MIST's own code.
- **Loader.** DALI against MIST's generic PyTorch loader.
- **Precision.** FP32 against BF16, if this GPU supports it.
- **Workers.** The generic loader with 4, 8, and 12 CPU workers (nnU-Net uses 12).

**Run it on an A100** (*Runtime > Change runtime type*), then *Runtime > Run all*. It
saves a CSV named after the GPU and downloads it. Each measurement runs in its own
process, and results are saved after each one.

What it leaves out: validation, checkpoints, and multi-GPU training. Synthetic files
on Colab's local disk may also read faster than real data on a network filesystem,
especially once they're in memory. The notebook reports how much memory the files
take, so you can judge how much of a real dataset would stay cached.

## Setup

In [ ]:
# Change MIST_REF to test a branch or commit.
MIST_REF = "main"
!pip install -q "mist-medical[dali] @ git+https://github.com/aecelaya/MIST.git@{MIST_REF}"

In [ ]:
import json
import os
import platform
import subprocess
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch

from mist.preprocessing import foreground_locations
from mist.utils import hardware, sitk_io

warnings.filterwarnings("ignore")

GIB = 1024**3

# Set MIST_LOADER_DRY_RUN to test this notebook's code on a CPU, with tiny data and
# only the generic loader. CPU timings say nothing about a GPU.
DRY_RUN = bool(os.environ.get("MIST_LOADER_DRY_RUN"))
if not (torch.cuda.is_available() or DRY_RUN):
    raise RuntimeError("No GPU found. Choose one under Runtime > Change runtime type.")

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "dry-run"
try:
    import nvidia.dali  # noqa: F401

    HAS_DALI = torch.cuda.is_available()
except ImportError:
    HAS_DALI = False

PRECISIONS = ["fp32"] + (["bf16"] if hardware.bf16_supported() else [])
CPUS = os.cpu_count()
RAM_GIB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / GIB
DATA_DIR = Path("mist_loading_data")
OUTPUT_CSV = f"mist_data_loading_{GPU_NAME.replace(' ', '_').replace('/', '-')}.csv"

print(f"GPU: {GPU_NAME}; {CPUS} CPUs; {RAM_GIB:.0f} GiB RAM")
print(f"PyTorch {torch.__version__}, Python {platform.python_version()}, DALI: {HAS_DALI}")
print(f"Precisions to measure: {PRECISIONS}")
print(f"Results go to {OUTPUT_CSV}")

## What to measure

The real LiTS fold trains on 104 cases, about 50 GiB of images after resampling.
The default here is 32 cases, about 16 GiB, which fits on Colab's disk and
generates in a few minutes. Raise `LITS_CASES` to see how a dataset larger than
memory behaves.

In [ ]:
LITS_CASES = 32
BRATS_CASES = 32
PATCH = [128, 128, 128]
BATCH = 2
NUM_WORKERS = 8  # training.hardware.num_cpu_workers in the LiTS run.
WORKER_COUNTS = [4, 8, 12]  # For the generic loader's sweep.
FOREGROUND_PROB = 0.6
SPACING = [0.767578125, 0.767578125, 1.0]
COMPILE = True

LOADER_WARMUP_BATCHES = 4
LOADER_BATCHES = 24
TRAIN_WARMUP_STEPS = 3  # Includes compiling, which isn't timed.
TRAIN_STEPS = 24
STEPS_PER_EPOCH = 250  # MIST's min_steps_per_epoch, for epoch estimates.
RUN_TIMEOUT_S = 1800

LITS_SIZE = (512, 512)  # x, y; z varies per case around 482.
LITS_Z_MEDIAN, LITS_Z_RANGE = 482, (250, 750)
BRATS_SIZE = (140, 172, 138)

if DRY_RUN:
    LITS_CASES = BRATS_CASES = 4
    PATCH = [32, 32, 32]
    NUM_WORKERS = 2
    WORKER_COUNTS = [1, 2]
    COMPILE = False
    LOADER_WARMUP_BATCHES, LOADER_BATCHES = 1, 3
    TRAIN_WARMUP_STEPS, TRAIN_STEPS = 1, 2
    LITS_SIZE = (96, 96)
    LITS_Z_MEDIAN, LITS_Z_RANGE = 80, (60, 100)
    BRATS_SIZE = (48, 56, 44)

## Make the data

Images are saved as MIST saves them: float32, `(x, y, z, channels)`, in C order.
*Old* labels go through MIST's conversion steps (`sitk_io.array_from_image`, then
`np.expand_dims` and `.astype(np.uint8)`) and are saved as they come out, as MIST did
before its fix. *New* labels are the same voxels saved in C order, with foreground
locations found and saved by MIST's own `foreground_locations` code, as
`mist_preprocess` now does. Each kind has its own folder, so old data has no saved
locations.

In [ ]:
def mist_label_array(label_zyx):
    """Turn a (z, y, x) label array into a saved label, the way MIST does."""
    image = sitk.GetImageFromArray(label_zyx)
    return np.expand_dims(sitk_io.array_from_image(image), axis=-1).astype(np.uint8)


def ellipsoid(shape, center, radii):
    """Return a boolean ellipsoid in a (z, y, x) grid."""
    z, y, x = (np.arange(n, dtype=np.float32) for n in shape)
    distance = (
        ((z - center[0]) / radii[0])[:, None, None] ** 2
        + ((y - center[1]) / radii[1])[None, :, None] ** 2
        + ((x - center[2]) / radii[2])[None, None, :] ** 2
    )
    return distance < 1.0


def lits_label(shape, rng):
    """A large organ with a few small lesions inside it."""
    label = np.zeros(shape, dtype=np.uint8)
    center = [s * rng.uniform(0.4, 0.6) for s in shape]
    radii = [s * rng.uniform(0.18, 0.25) for s in shape]
    label[ellipsoid(shape, center, radii)] = 1
    for _ in range(int(rng.integers(1, 6))):
        offset = [c + r * rng.uniform(-0.5, 0.5) for c, r in zip(center, radii, strict=True)]
        size = [max(2.0, s * rng.uniform(0.01, 0.04)) for s in shape]
        lesion = ellipsoid(shape, offset, size) & (label == 1)
        label[lesion] = 2
    return label


def brats_label(shape, rng):
    """Three nested tumor regions."""
    label = np.zeros(shape, dtype=np.uint8)
    center = [s * rng.uniform(0.35, 0.65) for s in shape]
    for value, scale in [(1, 0.22), (2, 0.14), (3, 0.07)]:
        radii = [s * scale * rng.uniform(0.8, 1.2) for s in shape]
        label[ellipsoid(shape, center, radii)] = value
    return label


def make_dataset(name, cases, channels, shape_fn, label_fn, seed):
    """Write images, MIST-made labels, and C-order label copies."""
    root = DATA_DIR / name
    folders = {
        "images": root / "images",
        "old": root / "old" / "labels",
        "new": root / "new" / "labels",
        "foreground": root / "new" / foreground_locations.FOLDER,
    }
    for folder in folders.values():
        folder.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    started = time.perf_counter()
    for case in range(cases):
        path = f"{name}_{case:03d}.npy"
        shape_xyz = shape_fn(rng)
        if (folders["foreground"] / path.replace(".npy", ".npz")).exists():
            continue
        image = np.empty((*shape_xyz, channels), dtype=np.float32)
        for c in range(channels):
            image[..., c] = rng.standard_normal(shape_xyz, dtype=np.float32)
        np.save(folders["images"] / path, image)
        del image
        label = mist_label_array(label_fn(shape_xyz[::-1], rng))
        np.save(folders["old"] / path, label)
        label = np.ascontiguousarray(label)
        np.save(folders["new"] / path, label)
        classes = [1, 2] if name == "lits" else [1, 2, 3]
        locations = foreground_locations.find(label, classes, np.random.default_rng(case))
        foreground_locations.save(
            folders["foreground"] / path.replace(".npy", ".npz"), locations, classes
        )
        del label
    elapsed = time.perf_counter() - started
    size = sum(f.stat().st_size for f in root.rglob("*.npy")) / GIB
    print(f"{name}: {cases} cases, {size:.1f} GiB on disk, made in {elapsed:.0f} s")


def lits_shape(rng):
    z = int(np.clip(rng.normal(LITS_Z_MEDIAN, 0.2 * LITS_Z_MEDIAN), *LITS_Z_RANGE))
    return (*LITS_SIZE, z)


def brats_shape(rng):
    return tuple(int(s + rng.integers(-8, 9)) for s in BRATS_SIZE)


make_dataset("lits", LITS_CASES, 1, lits_shape, lits_label, seed=0)
make_dataset("brats", BRATS_CASES, 4, brats_shape, brats_label, seed=1)

for name in ["lits", "brats"]:
    labels = sorted((DATA_DIR / name / "old" / "labels").glob("*.npy"))
    copies = sorted((DATA_DIR / name / "new" / "labels").glob("*.npy"))
    original, copy = np.load(labels[0], mmap_mode="r"), np.load(copies[0], mmap_mode="r")
    image = np.load(DATA_DIR / name / "images" / labels[0].name, mmap_mode="r")
    print(
        f"{name}: images C order: {image.flags['C_CONTIGUOUS']}; "
        f"old labels Fortran order: {original.flags['F_CONTIGUOUS'] and not original.flags['C_CONTIGUOUS']}; "
        f"new labels C order: {copy.flags['C_CONTIGUOUS']}; "
        f"same voxels: {np.array_equal(original, copy)}"
    )

## Measuring

This cell writes the script that runs each measurement in its own process. There
are three kinds:

- **loader:** MIST's training loader and GPU augmentation, built as MIST's trainer
  builds them, with no model. This is how fast data can arrive.
- **compute:** training steps on one batch kept on the GPU, so nothing waits for
  data. This is how fast the GPU can train.
- **train:** the full loop, as in `mist_train`. Each step is split into waiting for
  the next batch and the rest.

In [ ]:
WORKER = r"""
import json
import statistics
import sys
import time
import warnings

import torch

from mist.data_loading import augmentation, data_loader_registry
from mist.loss_functions.deep_supervision_wrapper import DeepSupervisionLoss
from mist.loss_functions.loss_registry import get_loss
from mist.models.model_registry import get_model_from_registry
from mist.utils import hardware


def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def build_loader(job):
    # As Patch3DTrainer.build_dataloaders builds it.
    settings = augmentation.settings_from_config(
        {"enabled": True, "transforms": {}}
    )
    loader = data_loader_registry.get_data_loader_from_registry(job["loader"])
    train_loader = loader.get_training_dataset(
        extract_patches=True,
        image_paths=job["images"],
        label_paths=job["labels"],
        dtm_paths=None,
        batch_size=job["batch"],
        oversampling=job["foreground_prob"],
        labels=job["classes"],
        roi_size=job["patch"],
        seed=42,
        num_workers=job["num_workers"],
        patch_margin=augmentation.spatial_margin(job["patch"], settings),
        rank=0,
        world_size=1,
    )
    return augmentation.AugmentedIterator(
        train_loader,
        augmentation.BatchAugmenter(patch_size=job["patch"], settings=settings, seed=42),
    )


def build_model(job, device):
    model = get_model_from_registry(
        "nnunet-plain",
        in_channels=job["channels"],
        out_channels=len(job["classes"]) + 1,
        patch_size=job["patch"],
        target_spacing=job["spacing"],
    ).to(device)
    if job["compile"]:
        model.compile()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4, eps=1e-8)
    criterion = DeepSupervisionLoss(get_loss("dice_ce")())
    return model, optimizer, criterion


def train_step(job, model, optimizer, criterion, batch):
    optimizer.zero_grad()
    with hardware.autocast_context(job["precision"] == "bf16"):
        output = model(batch["image"])
        loss = criterion(
            y_true=batch["label"],
            y_pred=output["prediction"],
            y_supervision=output.get("deep_supervision"),
            alpha=None,
            dtm=None,
        )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()


def next_batch(loader):
    started = time.perf_counter()
    batch = loader.next()[0]
    sync()
    return batch, time.perf_counter() - started


def measure(job):
    warnings.filterwarnings("ignore")
    hardware.tune_gpu_math()
    torch.manual_seed(0)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    result = {}

    started = time.perf_counter()
    loader = build_loader(job)
    first, result["first_batch_s"] = next_batch(loader)
    result["loader_setup_s"] = time.perf_counter() - started

    if job["kind"] == "loader":
        for _ in range(job["warmup"]):
            next_batch(loader)
        times = [next_batch(loader)[1] for _ in range(job["steps"])]
        result["seconds_per_batch"] = statistics.mean(times)
        result["median_seconds_per_batch"] = statistics.median(times)
        return result

    model, optimizer, criterion = build_model(job, device)
    waits, computes = [], []
    for step in range(job["warmup"] + job["steps"]):
        if job["kind"] == "compute":
            batch, wait = first, 0.0
        else:
            batch, wait = next_batch(loader)
        started = time.perf_counter()
        train_step(job, model, optimizer, criterion, batch)
        sync()
        if step >= job["warmup"]:
            waits.append(wait)
            computes.append(time.perf_counter() - started)
    result["wait_s"] = statistics.mean(waits)
    result["compute_s"] = statistics.mean(computes)
    result["seconds_per_step"] = result["wait_s"] + result["compute_s"]
    if torch.cuda.is_available():
        result["peak_reserved_gib"] = torch.cuda.max_memory_reserved() / 1024**3
    return result


if __name__ == "__main__":
    job = json.loads(sys.argv[1])
    try:
        result = {"status": "ok", **measure(job)}
    except Exception as error:  # Report it, and let the notebook go on.
        result = {"status": "error", "error": f"{type(error).__name__}: {error}"[-800:]}
    print("RESULT " + json.dumps(result))
"""
Path("loading_worker.py").write_text(WORKER)

In [ ]:
rows = []


def drop_page_cache():
    """Try to drop the OS file cache, so the next run reads from disk."""
    done = subprocess.run(
        "sync && echo 3 > /proc/sys/vm/drop_caches", shell=True, capture_output=True
    )
    return done.returncode == 0


def run(dataset, kind, loader="dali", layout="old", precision="fp32", cold=False, workers=None):
    """Run one measurement in its own process and save it."""
    root = DATA_DIR / dataset
    workers = workers or NUM_WORKERS
    names = sorted(p.name for p in (root / "images").glob("*.npy"))
    job = {
        "kind": kind,
        "loader": loader,
        "precision": precision,
        "images": [str(root / "images" / n) for n in names],
        "labels": [str(root / layout / "labels" / n) for n in names],
        "classes": [1, 2] if dataset == "lits" else [1, 2, 3],
        "channels": 1 if dataset == "lits" else 4,
        "patch": PATCH,
        "batch": BATCH,
        "spacing": SPACING,
        "foreground_prob": FOREGROUND_PROB,
        "num_workers": workers,
        "compile": COMPILE,
        "warmup": LOADER_WARMUP_BATCHES if kind == "loader" else TRAIN_WARMUP_STEPS,
        "steps": LOADER_BATCHES if kind == "loader" else TRAIN_STEPS,
    }
    dropped = drop_page_cache() if cold else False
    try:
        done = subprocess.run(
            [sys.executable, "loading_worker.py", json.dumps(job)],
            capture_output=True,
            text=True,
            timeout=RUN_TIMEOUT_S,
            check=False,
        )
        lines = [line for line in done.stdout.splitlines() if line.startswith("RESULT ")]
        result = (
            json.loads(lines[-1][len("RESULT ") :])
            if lines
            else {"status": "error", "error": done.stderr[-800:]}
        )
    except subprocess.TimeoutExpired:
        result = {"status": "timed out"}
    row = {
        "gpu": GPU_NAME,
        "cpus": CPUS,
        "ram_gib": round(RAM_GIB),
        "dataset": dataset,
        "cases": len(names),
        "kind": kind,
        "loader": loader if kind != "compute" else "",
        "data": layout if kind != "compute" else "",
        "workers": workers if kind != "compute" else "",
        "precision": precision if kind != "loader" else "",
        "cold_cache": dropped,
        **result,
    }
    rows.append(row)
    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
    if row["status"] != "ok":
        print(f"{dataset} {kind} {loader} {layout} {precision}: {row['status']}")
        print(f"    {row.get('error', '')}")
    elif kind == "loader":
        print(
            f"{dataset:5} loader  {loader:7} {layout} data, {workers} workers"
            f"{', cold' if dropped else ''}: {row['seconds_per_batch']:.3f} s/batch"
        )
    else:
        print(
            f"{dataset:5} {kind:7} {row['loader']:7} {row['data']:3} {row['workers']!s:2} {precision}: "
            f"{row['seconds_per_step']:.3f} s/step = wait {row['wait_s']:.3f} + compute "
            f"{row['compute_s']:.3f}; about {row['seconds_per_step'] * STEPS_PER_EPOCH / 60:.1f} "
            "min per epoch"
        )


LOADERS = (["dali"] if HAS_DALI else []) + ["generic"]
BEST = "bf16" if "bf16" in PRECISIONS else "fp32"

for dataset in ["lits", "brats"]:
    # How fast data arrives, for each loader, from old and new data.
    for loader in LOADERS:
        for layout in ["old", "new"]:
            run(dataset, "loader", loader, layout)
    # How fast the GPU trains when it never waits.
    # The loader only supplies the one batch, so any loader will do.
    for precision in PRECISIONS:
        run(dataset, "compute", "generic", "new", precision)
    # The full loop: the real LiTS run's settings first, then the fixes.
    for loader in LOADERS:
        for layout in ["old", "new"]:
            run(dataset, "train", loader, layout, "fp32")
    if BEST != "fp32" and HAS_DALI:
        run(dataset, "train", "dali", "new", BEST)
    # The generic loader with new data at the best precision, by worker count.
    for workers in WORKER_COUNTS:
        if workers <= CPUS:
            run(dataset, "train", "generic", "new", BEST, workers=workers)

# The same loader runs reading from disk, if Colab lets the cache be dropped.
if drop_page_cache():
    for loader in LOADERS:
        for layout in ["old", "new"]:
            run("lits", "loader", loader, layout, cold=True)
else:
    print("Couldn't drop the file cache, so every run read files already in memory.")

results = pd.DataFrame(rows)

## Results

*Wait* is time spent on `loader.next()`, the part of each step the GPU sits idle
waiting for data. *Compute* is the training step itself. The epoch estimate is
seconds per step × 250, without validation. Compare the first `train` row for LiTS,
which uses the real run's settings, with the 9 minutes per epoch it took on the
A100.

In [ ]:
ok = results[results["status"] == "ok"].copy()
loaders = ok[ok["kind"] == "loader"]
if len(loaders):
    print("Loader alone, seconds per batch of", BATCH)
    print(
        loaders.pivot_table(
            index=["dataset", "loader", "cold_cache"],
            columns="data",
            values="seconds_per_batch",
        )
        .round(3)
        .to_string()
    )

steps = ok[ok["kind"] != "loader"].copy()
if len(steps):
    steps["epoch_min"] = steps["seconds_per_step"] * STEPS_PER_EPOCH / 60
    columns = [
        "dataset",
        "kind",
        "loader",
        "data",
        "workers",
        "precision",
        "wait_s",
        "compute_s",
        "seconds_per_step",
        "epoch_min",
    ]
    print("\nTraining steps")
    print(steps[columns].round(3).to_string(index=False))

failed = results[results["status"] != "ok"]
if len(failed):
    print("\nDidn't finish:")
    print(failed[["dataset", "kind", "loader", "data", "status", "error"]].to_string())

## Save the results

Downloads the CSV. If the download doesn't start, it's in the Files panel on the
left.

In [ ]:
print(f"Saved {OUTPUT_CSV}")
try:
    from google.colab import files

    files.download(OUTPUT_CSV)
except ImportError:
    pass